# Plant Pathology Diagnosis — Train VGG16 on Plant Pathology 2020

**Run this notebook in [Google Colab](https://colab.research.google.com/) with the GPU runtime.**

Canonical training pipeline for the
[`Plant-Pathology-Diagnosis-VGG16`](https://github.com/MatamDinesh0802/Plant-Pathology-Diagnosis-VGG16) project.
It will:

1. Download the **Plant Pathology 2020** dataset (~1.5 GB) from Kaggle.
2. Fine-tune **VGG16** (ImageNet weights, custom head) on 4 disease classes.
3. Export the trained model to **ONNX** for portable inference.
4. Save metrics + figures for the Streamlit demo.

After training, download the artifacts and drop them into `models/` and `reports/`
of the GitHub repo. The Streamlit app picks them up automatically.

---

## 1. Setup — install packages

In [ ]:
!pip install -q tf2onnx onnx onnxruntime kaggle seaborn

## 2. Kaggle credentials

In [ ]:
from pathlib import Path

try:
    from google.colab import files
    print('Upload your kaggle.json (skip if already configured):')
    uploaded = files.upload()
    Path('~/.kaggle').expanduser().mkdir(exist_ok=True)
    !mv kaggle.json ~/.kaggle/ 2>/dev/null || true
    !chmod 600 ~/.kaggle/kaggle.json
except ImportError:
    print('Not on Colab — ensure ~/.kaggle/kaggle.json is configured.')

## 3. Download Plant Pathology 2020

⚠️ You must first accept the competition rules at
[kaggle.com/competitions/plant-pathology-2020-fgvc7/rules](https://www.kaggle.com/competitions/plant-pathology-2020-fgvc7/rules).

In [ ]:
!mkdir -p data/raw
!kaggle competitions download -c plant-pathology-2020-fgvc7 -p data/raw/
!unzip -q data/raw/plant-pathology-2020-fgvc7.zip -d data/raw/
!ls data/raw/ | head
!ls data/raw/images | wc -l

## 4. Imports + config

In [ ]:
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

print('TensorFlow', tf.__version__, '· GPU:', tf.config.list_physical_devices('GPU'))

DATA_DIR = Path('data/raw')
TRAIN_CSV = DATA_DIR / 'train.csv'
IMG_DIR = DATA_DIR / 'images'
OUT_DIR = Path('outputs'); OUT_DIR.mkdir(exist_ok=True)
FIG_DIR = OUT_DIR / 'figures'; FIG_DIR.mkdir(exist_ok=True)

CLASS_NAMES = ['healthy', 'multiple_diseases', 'rust', 'scab']
IMG_SIZE = 224
BATCH_SIZE = 32
RANDOM_STATE = 42

np.random.seed(RANDOM_STATE)
tf.random.set_seed(RANDOM_STATE)

## 5. Index dataset + class distribution

In [ ]:
df = pd.read_csv(TRAIN_CSV)
df['label_idx'] = df[CLASS_NAMES].values.argmax(axis=1)
df['label'] = df['label_idx'].map(dict(enumerate(CLASS_NAMES)))
df['path'] = df['image_id'].apply(lambda x: str(IMG_DIR / f'{x}.jpg'))
print(f'Total images: {len(df)}')
print(df['label'].value_counts())

fig, ax = plt.subplots(figsize=(8, 4))
counts = df['label'].value_counts().reindex(CLASS_NAMES)
colors = ['#10B981', '#F59E0B', '#EF4444', '#8B5CF6']
ax.bar(counts.index, counts.values, color=colors)
ax.set_title('Class distribution — Plant Pathology 2020')
ax.set_ylabel('Number of images')
for i, v in enumerate(counts.values):
    ax.text(i, v + 5, str(v), ha='center')
fig.tight_layout()
fig.savefig(FIG_DIR / 'class_distribution.png', dpi=150)
plt.show()

## 6. Sample images per class

In [ ]:
from PIL import Image

fig, axes = plt.subplots(2, 4, figsize=(14, 7))
for j, cls in enumerate(CLASS_NAMES):
    sample_paths = df[df['label'] == cls]['path'].head(2).tolist()
    for i, p in enumerate(sample_paths):
        img = Image.open(p).convert('RGB')
        axes[i, j].imshow(img)
        axes[i, j].axis('off')
        if i == 0:
            axes[i, j].set_title(cls, fontsize=12)
fig.tight_layout()
fig.savefig(FIG_DIR / 'sample_images.png', dpi=150)
plt.show()

## 7. Train / val / test split

In [ ]:
train_df, test_df = train_test_split(df, test_size=0.15, stratify=df['label_idx'], random_state=RANDOM_STATE)
train_df, val_df  = train_test_split(train_df, test_size=0.15, stratify=train_df['label_idx'], random_state=RANDOM_STATE)
print(f'train {len(train_df)} | val {len(val_df)} | test {len(test_df)}')

## 8. tf.data input pipelines

In [ ]:
AUTOTUNE = tf.data.AUTOTUNE
preprocess = tf.keras.applications.vgg16.preprocess_input

def decode_img(path, label, augment=False):
    img = tf.io.read_file(path)
    img = tf.image.decode_jpeg(img, channels=3)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    img = tf.cast(img, tf.float32)
    if augment:
        img = tf.image.random_flip_left_right(img)
        img = tf.image.random_brightness(img, 0.1)
        img = tf.image.random_contrast(img, 0.9, 1.1)
    img = preprocess(img)
    return img, label

def make_dataset(df_split, augment=False, shuffle=False):
    ds = tf.data.Dataset.from_tensor_slices((df_split['path'].values, df_split['label_idx'].values))
    if shuffle:
        ds = ds.shuffle(len(df_split), seed=RANDOM_STATE)
    ds = ds.map(lambda p, l: decode_img(p, l, augment=augment), num_parallel_calls=AUTOTUNE)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = make_dataset(train_df, augment=True, shuffle=True)
val_ds   = make_dataset(val_df)
test_ds  = make_dataset(test_df)

## 9. Build VGG16 (transfer learning)

In [ ]:
base = tf.keras.applications.VGG16(weights='imagenet', include_top=False,
                                    input_shape=(IMG_SIZE, IMG_SIZE, 3))
base.trainable = False

inputs = tf.keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3), name='image_input')
x = base(inputs, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.3)(x)
x = tf.keras.layers.Dense(128, activation='relu')(x)
x = tf.keras.layers.Dropout(0.3)(x)
outputs = tf.keras.layers.Dense(len(CLASS_NAMES), activation='softmax', name='class_proba')(x)
model = tf.keras.Model(inputs, outputs, name='vgg16_plant_pathology')
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
model.summary()

## 10. Train (head only) — fast

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(patience=6, restore_best_weights=True, monitor='val_accuracy'),
    tf.keras.callbacks.ReduceLROnPlateau(patience=3, factor=0.5, monitor='val_loss', min_lr=1e-7),
]
history1 = model.fit(train_ds, validation_data=val_ds, epochs=15, callbacks=callbacks)

## 11. Fine-tune (unfreeze last block of VGG16)

In [ ]:
# Unfreeze the last conv block (block5)
base.trainable = True
for layer in base.layers:
    if not layer.name.startswith('block5_'):
        layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy'],
)
history2 = model.fit(train_ds, validation_data=val_ds, epochs=10, callbacks=callbacks)

## 12. Training curves

In [ ]:
def join(h1, h2, key):
    return list(h1.history[key]) + list(h2.history[key])

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))
ax1.plot(join(history1, history2, 'loss'),     label='train', color='#16A34A')
ax1.plot(join(history1, history2, 'val_loss'), label='val',   color='#F43F5E')
ax1.set_title('Loss'); ax1.set_xlabel('epoch'); ax1.legend()
ax2.plot(join(history1, history2, 'accuracy'),     label='train', color='#16A34A')
ax2.plot(join(history1, history2, 'val_accuracy'), label='val',   color='#10B981')
ax2.set_title('Accuracy'); ax2.set_xlabel('epoch'); ax2.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / 'training_curves.png', dpi=150)
plt.show()

## 13. Evaluate on held-out test set

In [ ]:
y_test = []
y_pred = []
for batch_x, batch_y in test_ds:
    preds = model.predict(batch_x, verbose=0).argmax(axis=1)
    y_pred.extend(preds.tolist())
    y_test.extend(batch_y.numpy().tolist())
y_test = np.array(y_test); y_pred = np.array(y_pred)

acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred, average='weighted')
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=3)
report_dict = classification_report(y_test, y_pred, target_names=CLASS_NAMES, output_dict=True)
print(f'Test accuracy: {acc:.4f}  ·  weighted F1: {f1:.4f}\n')
print(report)

cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
ax.set_title('Confusion matrix — VGG16 on Plant Pathology 2020')
plt.setp(ax.get_xticklabels(), rotation=20, ha='right')
fig.tight_layout()
fig.savefig(FIG_DIR / 'confusion_matrix_vgg16.png', dpi=150)
plt.show()

## 14. Export — Keras + ONNX + metrics

In [ ]:
model.save(OUT_DIR / 'vgg16_model.keras')

import tf2onnx
spec = (tf.TensorSpec((None, IMG_SIZE, IMG_SIZE, 3), tf.float32, name='image_input'),)
onnx_model, _ = tf2onnx.convert.from_keras(model, input_signature=spec, opset=15)
with open(OUT_DIR / 'vgg16_model.onnx', 'wb') as f:
    f.write(onnx_model.SerializeToString())

metrics = {
    'best_model': 'vgg16',
    'n_train': int(len(train_df)),
    'n_val':   int(len(val_df)),
    'n_test':  int(len(test_df)),
    'epochs_phase1': len(history1.history['loss']),
    'epochs_phase2': len(history2.history['loss']),
    'models': {
        'vgg16': {
            'accuracy':    float(acc),
            'f1_weighted': float(f1),
            'report':      report_dict,
        },
    },
    'history': {
        'loss':     join(history1, history2, 'loss'),
        'val_loss': join(history1, history2, 'val_loss'),
        'accuracy': join(history1, history2, 'accuracy'),
        'val_accuracy': join(history1, history2, 'val_accuracy'),
    },
}
with open(OUT_DIR / 'metrics.json', 'w') as f:
    json.dump(metrics, f, indent=2)
print('Saved:', sorted(os.listdir(OUT_DIR)))

## 15. Download artifacts

Drop these into the GitHub repo:

| Colab file | Destination |
|---|---|
| `outputs/vgg16_model.onnx` | `models/vgg16_model.onnx` |
| `outputs/metrics.json` | `reports/metrics.json` |
| `outputs/figures/*.png` | `reports/figures/` |

Then `streamlit run app/streamlit_app.py`.

In [ ]:
import shutil
shutil.make_archive('plant_pathology_artifacts', 'zip', OUT_DIR)

try:
    from google.colab import files
    files.download('plant_pathology_artifacts.zip')
except ImportError:
    print('Not on Colab — artifacts are in outputs/.')